# 🎧 Stage 5 — Computer Audio Basics: Interactive Audio Analysis Lab
### *Upload an Audio File and Explore Its Digital Representation*

Welcome! This notebook is a hands-on lab for understanding **how computers represent sound**.

You will:
1. Upload your own audio file (`.wav`, `.mp3`, `.flac`, `.m4a`)
2. Automatically analyze it
3. See visualizations of every core "Computer Audio Basics" concept
4. Get short, beginner-friendly explanations before and after every plot

**How to use this notebook:** Click `Runtime → Run all` in Google Colab, then upload your audio file when prompted in Section 2. Everything after that runs automatically on **your** audio — there are no fake/example numbers anywhere in this notebook.

> This is Stage 5. Topics like FFT, STFT, Spectrogram, and Mel-Spectrogram are only **previewed** here — they are the subject of the *next* stage.


## Section 1 — Install & Import Libraries

We need a small set of Python libraries:

| Library | What it's for |
|---|---|
| `librosa` | Loading audio, computing audio features (RMS, pitch, FFT helpers) |
| `soundfile` | Reading audio file metadata (like bit depth) |
| `numpy` | Numerical arrays — audio is just numbers! |
| `matplotlib` | Plotting waveforms and spectra |
| `scipy` | Signal processing helpers |
| `pandas` | Clean summary tables |
| `IPython.display.Audio` | An in-notebook audio player |

Run the cell below once. In Google Colab, `librosa` and `soundfile` are not always pre-installed, so we install them quietly if missing.


In [ ]:
# Install packages if they are missing (safe to re-run)
import importlib
import subprocess
import sys

required = ["librosa", "soundfile", "numpy", "matplotlib", "scipy", "pandas"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Installing missing packages: {missing} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Done installing.")
else:
    print("All required packages are already available.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf

from scipy import signal
from IPython.display import Audio, display, Markdown

# Detect whether we're actually running inside Google Colab.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Note: google.colab not found — you are not running in Colab.")
    print("Section 2 will fall back to a manual file-path input instead of the upload widget.")

# Clean, readable default plotting style (no custom colors forced)
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

print("Libraries imported successfully.")


## Section 2 — Upload Your Audio File

Click the button that appears after running the cell below, then choose a `.wav`, `.mp3`, `.flac`, or `.m4a` file from your computer.

**Why these formats?** `librosa` decodes audio through `soundfile`/`audioread`, which reliably supports WAV, FLAC, and OGG natively, and MP3/M4A on most systems (Colab has the needed backends preinstalled). If a format truly can't be decoded, this notebook will tell you clearly instead of failing silently — the usual fix is converting the file to WAV first (e.g. with `ffmpeg -i input.ext output.wav`).


In [ ]:
import os

audio_file = None

if IN_COLAB:
    print("Please choose an audio file to upload (.wav, .mp3, .flac, .m4a) ...")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise RuntimeError("No file was uploaded. Please re-run this cell and select a file.")
    audio_file = list(uploaded.keys())[0]
else:
    # Fallback for non-Colab environments
    audio_file = input("Enter the path to your audio file: ").strip()

if not os.path.exists(audio_file):
    raise FileNotFoundError(f"Could not find the file '{audio_file}'.")

file_ext = os.path.splitext(audio_file)[1].lower()
file_size_bytes = os.path.getsize(audio_file)
file_size_kb = file_size_bytes / 1024

supported_exts = [".wav", ".mp3", ".flac", ".m4a", ".ogg"]

print("=" * 50)
print("UPLOADED FILE INFO")
print("=" * 50)
print(f"Filename       : {audio_file}")
print(f"File extension : {file_ext}")
print(f"File size      : {file_size_kb:.2f} KB ({file_size_bytes} bytes)")

if file_ext not in supported_exts:
    print()
    print(f"Warning: '{file_ext}' is not in the commonly-supported list {supported_exts}.")
    print("If loading fails below, convert the file to WAV first, e.g.:")
    print("    ffmpeg -i your_file INPUT_EXT your_file.wav")


## Section 3 — Play the Audio

Before we analyze anything numerically, let's **listen** to the file.

> The waveform (coming up in Section 8) is a *visual* representation of the signal. This audio widget below lets you *hear* the very same signal — it's useful to connect what you see with what you hear.


In [ ]:
print("Play Uploaded Audio:")
display(Audio(audio_file))


## Section 4 — Load the Audio (Preserving Channels & Sample Rate)

### Concept
To analyze audio, we need to turn the file into numbers. `librosa.load()` decodes the file into a NumPy array of samples.

- **`sr` (sampling rate)** — how many samples were taken per second of sound
- **`audio`** — the actual digital samples (the numeric representation of the waveform)
- **`sr=None`** — keep the file's *original* sampling rate (don't resample)
- **`mono=False`** — keep original channels (don't force stereo → mono yet), so we can study channels honestly in Section 7

### What we are doing in code
We call `librosa.load` with `sr=None, mono=False`, then print the key facts about what we loaded.


In [ ]:
try:
    audio, sr = librosa.load(audio_file, sr=None, mono=False)
except Exception as e:
    raise RuntimeError(
        f"Could not decode '{audio_file}' with librosa. "
        f"Try converting it to WAV first (e.g. with ffmpeg). Original error: {e}"
    )

# Normalize array shape bookkeeping: librosa gives (samples,) for mono
# and (channels, samples) for multi-channel audio.
is_stereo = (audio.ndim == 2)
n_channels = audio.shape[0] if is_stereo else 1
n_samples = audio.shape[-1]
duration_sec = n_samples / sr

print("=" * 50)
print("LOADED AUDIO INFO")
print("=" * 50)
print(f"Sampling rate     : {sr} Hz")
print(f"Number of samples : {n_samples}")
print(f"Number of channels: {n_channels} ({'Stereo' if is_stereo else 'Mono'})")
print(f"Duration          : {duration_sec:.3f} seconds")


## Section 5 — Audio Summary Dashboard

### Concept
Before diving into individual concepts, let's build one clean table summarizing everything librosa and soundfile can tell us about the file — all computed directly from **your** upload, nothing pre-filled.

### Research connection
This is exactly the kind of metadata table researchers log for every file in a dataset (e.g. a speech corpus), so that dataset statistics and biases (like inconsistent sample rates) can be caught early.


In [ ]:
# Try to get reliable format-level info (bit depth / subtype) from soundfile
bit_depth_str = "Bit depth could not be reliably determined from this decoding pipeline."
file_format = file_ext.replace(".", "").upper()
try:
    info = sf.info(audio_file)
    file_format = info.format
    if info.subtype:
        bit_depth_str = info.subtype  # e.g. 'PCM_16', 'PCM_24', 'FLOAT'
except Exception:
    pass  # e.g. MP3 often isn't readable by soundfile directly; that's OK, we say so above

min_amp = float(np.min(audio))
max_amp = float(np.max(audio))
mean_amp = float(np.mean(audio))
rms_amp = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))

summary_rows = [
    ("Filename", audio_file, "Name of the uploaded file"),
    ("File format", file_format, "Container/codec of the audio file"),
    ("File size", f"{file_size_kb:.2f} KB", "Size of the file on disk"),
    ("Sampling rate", f"{sr} Hz", "Samples captured per second"),
    ("Duration", f"{duration_sec:.3f} s", "Length of the audio"),
    ("Number of channels", n_channels, "1 = mono, 2 = stereo"),
    ("Number of samples", n_samples, "Total digital samples per channel"),
    ("Bit depth / subtype", bit_depth_str, "Precision used to store each sample"),
    ("Minimum amplitude", f"{min_amp:.5f}", "Smallest (most negative) sample value"),
    ("Maximum amplitude", f"{max_amp:.5f}", "Largest (most positive) sample value"),
    ("Mean amplitude", f"{mean_amp:.6f}", "Average sample value (near 0 for most audio)"),
    ("RMS amplitude", f"{rms_amp:.5f}", "Root-mean-square — a measure of signal energy"),
]

summary_df = pd.DataFrame(summary_rows, columns=["Property", "Value", "Meaning"])
summary_df


## Section 6 — What Is Digital Audio?

### Concept
Real sound is a continuous physical vibration in the air. Computers can't store "continuous" — they store **numbers**. Digital audio is the result of this pipeline:

```
Real-world sound
      |
  Microphone           (converts air pressure -> electrical voltage)
      |
 Analog signal          (continuous electrical wave)
      |
   Sampling             (measuring the voltage at fixed time intervals)
      |
 Digital samples        (a list of numbers)
      |
  Audio file            (numbers saved to disk in a format like WAV/MP3)
```

The computer never sees a smooth curve — it only ever sees a long list of numbers. Let's look at the literal numbers from your uploaded audio.


In [ ]:
# Show the first samples of the uploaded audio as raw numbers
first_channel = audio[0] if is_stereo else audio
n_preview = min(50, len(first_channel))
preview_samples = first_channel[:n_preview]

print(f"First {n_preview} raw sample values of the uploaded audio:")
print(np.round(preview_samples, 5).tolist())


**What this means:** every one of those numbers is one *measurement* of the sound wave's pressure/voltage at one instant in time. The whole file is just millions of these numbers in a row — that's what "digital audio" really is.

## Section 7 — Audio Shape & Channels

### Concept
- **Mono** = one channel -> array shape `(samples,)`
- **Stereo** = two channels (Left, Right) -> array shape `(2, samples)`

librosa reports the shape of whatever we loaded, so we can tell exactly which case we're in — no guessing.


In [ ]:
print(f"Array shape: {audio.shape}")

if is_stereo:
    print(f"-> This is STEREO audio: {n_channels} channels, {audio.shape[1]} samples each.")
    left = audio[0]
    right = audio[1] if n_channels > 1 else audio[0]

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    librosa.display.waveshow(left, sr=sr, ax=axes[0])
    axes[0].set_title("Left Channel Waveform")
    axes[0].set_ylabel("Amplitude")

    librosa.display.waveshow(right, sr=sr, ax=axes[1])
    axes[1].set_title("Right Channel Waveform")
    axes[1].set_xlabel("Time (seconds)")
    axes[1].set_ylabel("Amplitude")

    plt.tight_layout()
    plt.show()
else:
    print("-> This is MONO audio: 1 channel.")
    plt.figure(figsize=(12, 3))
    librosa.display.waveshow(audio, sr=sr)
    plt.title("Mono Waveform")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.show()


## Section 8 — Waveform Visualization

### Concept
A **waveform** plots amplitude (y-axis) against time (x-axis). It's the most fundamental way to "see" sound.

- **X-axis** = time in seconds
- **Y-axis** = amplitude (roughly, how far the signal deviates from silence — 0)
- **Positive amplitude** = pressure above the resting point
- **Negative amplitude** = pressure below the resting point
- **Zero crossing** = a point where the wave crosses from positive to negative (or vice versa) — useful for detecting periodicity
- **Large-amplitude regions** = louder / more energetic moments
- **Small-amplitude regions** = quieter moments, possibly silence

For stereo files we use the first channel here so the whole waveform fits cleanly on one plot (full L/R comparison is in Section 7/22).


In [ ]:
wave_for_plot = audio[0] if is_stereo else audio

plt.figure(figsize=(14, 4))
librosa.display.waveshow(wave_for_plot, sr=sr)
plt.title("Waveform of Uploaded Audio")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.axhline(0, linewidth=0.8)
plt.show()


## Section 9 — Zoom Into a Small Segment (Discrete Samples)

### Concept
This is the single most important visual for understanding digital audio: a computer **never** stores a smooth curve. It stores individual, discrete sample *points*. Zoomed far out, those points blur together and look continuous. Zoomed way in, you can see the individual dots.

### What we are doing in code
We take the first ~0.05 seconds of audio and plot it two ways: as a connected line (looks continuous) and with the individual sample points marked (clearly discrete).


In [ ]:
zoom_duration = 0.05  # seconds
zoom_samples = int(min(zoom_duration * sr, len(wave_for_plot)))
zoom_samples = max(zoom_samples, min(50, len(wave_for_plot)))  # ensure at least a few points on very short/low-sr audio

segment = wave_for_plot[:zoom_samples]
t = np.arange(len(segment)) / sr

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(t, segment, linewidth=1)
axes[0].set_title("Looks Continuous (line plot)")
axes[0].set_xlabel("Time (seconds)")
axes[0].set_ylabel("Amplitude")

axes[1].plot(t, segment, linewidth=1, alpha=0.5)
axes[1].scatter(t, segment, s=12)
axes[1].set_title(f"Actually Discrete: {len(segment)} Individual Samples")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

print(f"Displayed {len(segment)} discrete samples spanning {len(segment)/sr*1000:.2f} ms.")


**What this means:** the left plot and right plot show the *exact same data*. The only difference is whether we connect the dots. A computer's actual audio file is the right-hand picture — a finite list of discrete numbers — the smooth look on the left is just how our eyes/matplotlib interpolate between them.

## Section 10 — Sampling Rate

### Concept
**Sampling rate** = how many samples are captured per second of audio, measured in Hz (samples/second). Example: 16 kHz = 16,000 samples every second.

### Mathematical idea
$$\text{Nyquist frequency} = \frac{\text{sampling rate}}{2}$$

This comes from the **Nyquist–Shannon sampling theorem**: to faithfully represent a frequency $f$, you need to sample at *at least* $2f$ times per second. So the highest frequency a given sampling rate can represent is half the sampling rate.

### Research connection
Speech is usually well-represented by 16 kHz sampling (Nyquist ~= 8 kHz, which covers almost all speech-relevant frequencies). Music/general audio often uses 44.1 kHz or 48 kHz to capture higher frequencies.


In [ ]:
samples_per_ms = sr / 1000
samples_per_sec = sr
samples_per_min = sr * 60
nyquist = sr / 2

print(f"Sampling rate            : {sr} Hz")
print(f"Samples per millisecond  : {samples_per_ms:.2f}")
print(f"Samples per second       : {samples_per_sec}")
print(f"Samples per minute       : {samples_per_min}")
print(f"Nyquist frequency        : {nyquist} Hz")
print()
print(f"-> This audio can faithfully represent frequencies up to about {nyquist:.0f} Hz.")


## Section 11 — Nyquist / Aliasing Demonstration (Synthetic Example)

> **Concept demonstration using synthetic signals** — everything in this section is generated sine waves, *not* your uploaded audio. This is purely to teach the sampling theorem.

### Concept
If we try to sample a frequency **above** the Nyquist limit, the samples can't represent it correctly — the reconstructed signal looks like a *different, lower* frequency. This is called **aliasing**, and it's why sampling rate matters so much.


In [ ]:
# --- Synthetic demonstration only (not the uploaded audio) ---
demo_sr = 100  # a deliberately low sampling rate, in Hz, for a clear teaching example
demo_nyquist = demo_sr / 2
t_continuous = np.linspace(0, 1, 5000)  # a fine timeline to fake "continuous"

freqs_demo = {
    "Low frequency (5 Hz) - well below Nyquist": 5,
    f"Near Nyquist ({demo_nyquist:.0f} Hz)": demo_nyquist,
    "Above Nyquist (80 Hz) - will ALIAS": 80,
}

fig, axes = plt.subplots(len(freqs_demo), 1, figsize=(12, 9))
for ax, (label, f) in zip(axes, freqs_demo.items()):
    continuous_wave = np.sin(2 * np.pi * f * t_continuous)
    ax.plot(t_continuous, continuous_wave, alpha=0.4, label="'True' continuous signal")

    t_sampled = np.arange(0, 1, 1 / demo_sr)
    sampled_wave = np.sin(2 * np.pi * f * t_sampled)
    ax.plot(t_sampled, sampled_wave, "o-", label=f"Sampled at {demo_sr} Hz")

    ax.set_title(f"{label}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

print("Notice how the 80 Hz wave, once sampled at only 100 Hz, produces sample points")
print("that could just as easily belong to a much slower wave. That false slow wave")
print("is the 'alias' - this is exactly why sampling rate must be chosen carefully.")


## Section 12 — Amplitude

### Concept
**Amplitude** is the value of a single audio sample — how far the signal deviates from zero at that instant. It reflects signal strength, but it is **not exactly the same as perceived loudness** (human loudness perception is logarithmic and frequency-dependent — that's a separate topic, decibels/psychoacoustics).


In [ ]:
print(f"Minimum amplitude : {min_amp:.5f}")
print(f"Maximum amplitude  : {max_amp:.5f}")
print(f"Mean amplitude     : {mean_amp:.6f}")
print(f"RMS amplitude      : {rms_amp:.5f}")

# Plot a short section (first 1 second or full length if shorter) to inspect amplitude visually
short_len = min(sr, len(wave_for_plot))
plt.figure(figsize=(12, 3))
librosa.display.waveshow(wave_for_plot[:short_len], sr=sr)
plt.title("Amplitude Over a Short Section (first 1 second)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.show()


## Section 13 — RMS Energy

### Concept
**RMS (Root Mean Square)** summarizes signal *energy* over a short window, smoothing out the raw amplitude into something closer to perceived loudness/strength over time.

### Mathematical idea
$$\text{RMS} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} x_i^2}$$

### Research connection
RMS is a workhorse feature in speech processing — used for silence/voice-activity detection, loudness normalization, and as an input feature to many audio models.


In [ ]:
rms_mono = librosa.feature.rms(y=wave_for_plot)[0]
rms_times = librosa.frames_to_time(np.arange(len(rms_mono)), sr=sr)

plt.figure(figsize=(12, 4))
plt.plot(rms_times, rms_mono)
plt.title("RMS Energy Over Time")
plt.xlabel("Time (seconds)")
plt.ylabel("RMS Energy")
plt.show()


## Section 14 — Silence Detection

### Concept
We can estimate which parts of the audio are "non-silent" using an energy threshold via `librosa.effects.split()`.

> Silence detection **depends on a threshold** and is not perfect. A quiet breath, background hum, or a soft consonant can be misclassified. Treat this as an estimate, not ground truth.


In [ ]:
intervals = librosa.effects.split(wave_for_plot, top_db=30)

non_silent_samples = sum((end - start) for start, end in intervals)
silent_samples = n_samples - non_silent_samples
non_silent_pct = 100 * non_silent_samples / n_samples if n_samples else 0
silent_pct = 100 - non_silent_pct

print(f"Total duration            : {duration_sec:.3f} s")
print(f"Estimated non-silent time : {non_silent_samples/sr:.3f} s ({non_silent_pct:.1f}%)")
print(f"Estimated silent time     : {silent_samples/sr:.3f} s ({silent_pct:.1f}%)")

plt.figure(figsize=(14, 4))
librosa.display.waveshow(wave_for_plot, sr=sr, alpha=0.5)
for start, end in intervals:
    plt.axvspan(start / sr, end / sr, color="green", alpha=0.15)
plt.title("Waveform with Estimated Non-Silent Regions Highlighted (green)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.show()


## Section 15 — Noise

### Concept
**Noise** is any unwanted component mixed into the signal we care about. This notebook cannot perfectly *identify* noise in your specific upload automatically — that requires more advanced modeling. What we *can* do honestly:

1. Show basic signal statistics (already computed above)
2. Point out low-energy regions (from the RMS plot in Section 13)
3. Explain that background noise can be present even in "silent" regions
4. Show an **optional synthetic demonstration** of signal + noise

> The demo below uses a *synthetic* sine wave, not your uploaded audio, so we can show a clean ground truth.


In [ ]:
# --- Synthetic educational example: clean signal + Gaussian noise ---
demo_duration = 1.0
demo_sr2 = 8000
t_demo = np.linspace(0, demo_duration, int(demo_sr2 * demo_duration), endpoint=False)
clean_signal = 0.5 * np.sin(2 * np.pi * 220 * t_demo)  # a clean 220 Hz tone

noise = np.random.normal(0, 0.15, size=clean_signal.shape)
noisy_signal = clean_signal + noise

signal_power = np.mean(clean_signal ** 2)
noise_power = np.mean(noise ** 2)
snr_db = 10 * np.log10(signal_power / noise_power)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
axes[0].plot(t_demo[:400], clean_signal[:400])
axes[0].set_title("Synthetic CLEAN Signal (220 Hz tone)")
axes[0].set_ylabel("Amplitude")

axes[1].plot(t_demo[:400], noisy_signal[:400])
axes[1].set_title("Synthetic NOISY Signal (clean + Gaussian noise)")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

print(f"Synthetic-demo SNR: {snr_db:.2f} dB")
print("Observed signal = desired signal + noise -> exactly what you see in the bottom plot above.")


## Section 16 — Frequency

### Concept
**Frequency** = number of full oscillation cycles per second, measured in **Hz**. Higher frequency = faster oscillation = (usually) a higher-pitched sound.

Below is a synthetic comparison of three frequencies so the visual difference is obvious.


In [ ]:
demo_sr3 = 5000
demo_freqs = [10, 100, 1000]
window_lengths = [0.5, 0.05, 0.01]  # shorter windows for higher frequencies so cycles are visible

fig, axes = plt.subplots(len(demo_freqs), 1, figsize=(12, 8))
for ax, f, win in zip(axes, demo_freqs, window_lengths):
    t_f = np.linspace(0, win, int(demo_sr3 * win))
    wave = np.sin(2 * np.pi * f * t_f)
    ax.plot(t_f, wave)
    ax.set_title(f"{f} Hz sine wave (showing first {win*1000:.0f} ms)")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

print("Notice: as frequency increases, more cycles fit into the same amount of time.")


## Section 17 — Frequency Content of Uploaded Audio (Preview: Frequency-Domain Analysis)

> **Preview for Next Stage.** FFT/STFT/Spectrogram belong to Stage 6. This is just a first glimpse.

### Concept
The **waveform** tells us amplitude vs. time. The **FFT (Fast Fourier Transform)** tells us, approximately, *which frequencies are present* in a segment of audio — amplitude vs. frequency instead of amplitude vs. time.

### What we are doing in code
We take one short windowed segment of your uploaded audio (not the entire file) and compute its FFT magnitude spectrum.


In [ ]:
fft_window_sec = min(1.0, duration_sec)
fft_window_samples = int(fft_window_sec * sr)
fft_segment = wave_for_plot[:fft_window_samples]

fft_result = np.fft.rfft(fft_segment)
fft_magnitude = np.abs(fft_result)
fft_freqs = np.fft.rfftfreq(len(fft_segment), d=1/sr)

plt.figure(figsize=(12, 4))
plt.plot(fft_freqs, fft_magnitude)
plt.title(f"FFT Magnitude Spectrum (first {fft_window_sec:.2f}s of uploaded audio)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.xlim(0, nyquist)
plt.show()

dominant_freq = fft_freqs[np.argmax(fft_magnitude)]
print(f"Dominant frequency component in this segment: ~{dominant_freq:.1f} Hz")


## Section 18 — Fundamental Frequency Preview (Optional, Speech Only)

### Concept
**F0 (fundamental frequency)** relates to perceived *pitch* in voiced speech or musical tones. We attempt to estimate it with `librosa.pyin`, a robust pitch-tracking algorithm.

> Pitch estimation can fail or be meaningless on noise, silence, unvoiced speech (like "s", "f", "sh"), music with many overlapping notes, or very short clips. If it fails here, we will say so — not fabricate a number.


In [ ]:
try:
    f0, voiced_flag, voiced_probs = librosa.pyin(
        wave_for_plot.astype(np.float32),
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr,
    )
    valid_f0 = f0[~np.isnan(f0)]
    if len(valid_f0) > 0:
        print(f"Estimated F0 range : {valid_f0.min():.1f} Hz - {valid_f0.max():.1f} Hz")
        print(f"Estimated F0 mean  : {valid_f0.mean():.1f} Hz")
        print(f"Voiced frames       : {np.sum(voiced_flag)} / {len(voiced_flag)}")

        times_f0 = librosa.times_like(f0, sr=sr)
        plt.figure(figsize=(12, 4))
        plt.plot(times_f0, f0)
        plt.title("Estimated Pitch (F0) Over Time")
        plt.xlabel("Time (seconds)")
        plt.ylabel("Frequency (Hz)")
        plt.show()
    else:
        print("No voiced pitch could be reliably detected in this audio.")
        print("This is expected for music, noise, silence, or unvoiced-only speech.")
except Exception as e:
    print(f"Pitch estimation failed: {e}")
    print("This can happen on very short clips or unusual audio content.")


## Section 19 — Audio Duration

### Mathematical idea
$$\text{Duration} = \frac{N}{F_s}$$
where $N$ = number of samples, $F_s$ = sampling rate.


In [ ]:
computed_duration = n_samples / sr
print(f"N (samples)        = {n_samples}")
print(f"Fs (sampling rate) = {sr} Hz")
print(f"Duration = N / Fs = {n_samples} / {sr} = {computed_duration:.3f} seconds")


## Section 20 — Bit Depth

### Concept
**Bit depth** is how many bits are used to store each individual sample's amplitude value — it determines the number of possible amplitude *levels*.

| Bit depth | Possible levels |
|---|---|
| 8-bit | $2^8$ = 256 |
| 16-bit | $2^{16}$ = 65,536 |
| 24-bit | $2^{24}$ = 16,777,216 |
| 32-bit float | Effectively continuous within a normalized range |

Higher bit depth = finer amplitude resolution = less quantization noise/distortion.

> We will **not** guess bit depth from a normalized floating-point array (which `librosa.load` always returns) — instead we read it from the original file's metadata via `soundfile`, when possible.


In [ ]:
for bits in [8, 16, 24]:
    print(f"{bits}-bit -> 2^{bits} = {2**bits:,} possible amplitude levels")

print()
print(f"This file's reported subtype/bit depth: {bit_depth_str}")

if "PCM_16" in bit_depth_str:
    print("-> PCM_16 means 16-bit linear PCM: 65,536 possible amplitude levels per sample.")
elif "PCM_24" in bit_depth_str:
    print("-> PCM_24 means 24-bit linear PCM: over 16.7 million possible amplitude levels.")
elif "FLOAT" in bit_depth_str:
    print("-> FLOAT means samples are stored as floating-point numbers, not fixed integer levels.")


## Section 21 — WAV vs MP3

| | WAV | MP3 |
|---|---|---|
| Compression | Typically none (raw PCM) | Lossy (perceptual) compression |
| Lossless/lossy | Lossless | Lossy — some information is permanently discarded |
| File size | Large | Much smaller |
| Audio quality | Highest fidelity to the original samples | Slightly degraded, especially at low bitrates |
| Research considerations | Preferred for reproducible analysis | Can introduce **compression artifacts** |

### Research connection — a dataset confound
If, say, all your **human** speech recordings are WAV and all your **synthetic/AI** speech recordings are MP3 (or vice versa), a classifier trained on this data might learn to detect *compression artifacts* rather than genuine human-vs-AI speech characteristics. Always keep file formats/encodings **consistent and balanced** across classes in audio research.


## Section 22 — Channel Analysis

If the uploaded audio is stereo, we compare the Left and Right channels directly. If mono, there's only one channel to report on.


In [ ]:
if is_stereo:
    left_rms = float(np.sqrt(np.mean(audio[0].astype(np.float64) ** 2)))
    right_rms = float(np.sqrt(np.mean(audio[1].astype(np.float64) ** 2))) if n_channels > 1 else left_rms

    print(f"Left channel RMS  : {left_rms:.5f}")
    print(f"Right channel RMS : {right_rms:.5f}")
    diff_pct = 100 * abs(left_rms - right_rms) / max(left_rms, right_rms, 1e-12)
    print(f"Relative difference: {diff_pct:.1f}%")
    if diff_pct < 5:
        print("-> Left and Right channels are very similar in energy (likely a centered/mono-like mix).")
    else:
        print("-> Left and Right channels differ noticeably in energy (a genuinely stereo mix).")
else:
    print("This audio is MONO - there is only one channel, so no left/right comparison applies.")


## Section 23 — Mono Conversion Demonstration

### Concept
Many speech models expect **mono** input. A common way to combine stereo into mono is averaging the channels: `mono = (left + right) / 2`.

We create a **new, separate array** for this — the original uploaded audio is never overwritten.

### Research connection
Datasets are frequently standardized to mono so that channel count doesn't become an accidental confound (similar to the WAV/MP3 issue in Section 21).


In [ ]:
if is_stereo:
    original_left = audio[0].copy()
    original_right = audio[1].copy() if n_channels > 1 else audio[0].copy()
    converted_mono = (original_left + original_right) / 2.0

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, sharey=True)
    librosa.display.waveshow(original_left, sr=sr, ax=axes[0])
    axes[0].set_title("Original Left")
    librosa.display.waveshow(original_right, sr=sr, ax=axes[1])
    axes[1].set_title("Original Right")
    librosa.display.waveshow(converted_mono, sr=sr, ax=axes[2])
    axes[2].set_title("Converted Mono (average of L and R)")
    axes[2].set_xlabel("Time (seconds)")
    plt.tight_layout()
    plt.show()
else:
    print("This audio is already mono - no conversion needed.")
    converted_mono = wave_for_plot.copy()


## Section 24 — Audio Normalization

### Concept
**Normalization** rescales amplitude so the loudest sample reaches a target level (commonly +/-1.0), without changing the *timing* structure of the signal — only its overall scale.

We create a normalized **copy**; the original uploaded audio is left untouched.


In [ ]:
max_abs_amp = float(np.max(np.abs(wave_for_plot)))
print(f"Maximum absolute amplitude before normalization: {max_abs_amp:.5f}")

if max_abs_amp > 0:
    normalized_audio = wave_for_plot / max_abs_amp
else:
    normalized_audio = wave_for_plot.copy()
    print("Audio appears to be silent (max amplitude is 0) - normalization skipped.")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
librosa.display.waveshow(wave_for_plot, sr=sr, ax=axes[0])
axes[0].set_title("Original Waveform")
librosa.display.waveshow(normalized_audio, sr=sr, ax=axes[1])
axes[1].set_title("Normalized Waveform (peak scaled to +/-1.0)")
axes[1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

print(f"New maximum absolute amplitude after normalization: {float(np.max(np.abs(normalized_audio))):.5f}")
print("Note: the timing/shape of the wave is unchanged - only its scale.")


## Section 25 — Speech Signal Analysis (Summary View)

If your uploaded file contains speech, here is everything computed so far, side by side, to show how each representation adds information:

1. **Waveform** (Section 8) — raw amplitude over time
2. **RMS energy** (Section 13) — where the signal is loud/quiet
3. **Silence/non-silence** (Section 14) — where speech is actually present
4. **Frequency spectrum** (Section 17 preview) — which frequencies are active
5. **Pitch / F0** (Section 18, if detected) — perceived pitch contour

```
Speech
  |
Waveform
  |
Energy (RMS)
  |
Frequency (FFT)
  |
Pitch (F0)
  |
Features
  |
Machine Learning / Deep Learning
```

Each step extracts a more compact, more *meaningful* representation than the raw waveform — this pipeline is exactly the foundation for feature extraction in speech ML systems.


## Section 26 — Important Audio Concept Table

In [ ]:
concept_table = pd.DataFrame([
    ["Digital Audio", "Sound represented as a sequence of numbers", "-", "A .wav file's sample array", "Foundation of all audio computing"],
    ["Sampling", "Measuring a signal's value at regular time intervals", "-", "Taking 16,000 measurements per second", "Determines what frequencies can be captured"],
    ["Sampling Rate", "Number of samples taken per second", "Hz", "16000 Hz, 44100 Hz", "Sets the Nyquist limit for usable frequencies"],
    ["Bit Depth", "Bits used to store each sample's amplitude", "bits", "16-bit -> 65,536 levels", "Affects dynamic range and quantization noise"],
    ["Amplitude", "Instantaneous value of a sample", "unitless / dB", "0.42, -0.15", "Relates to signal strength, not exactly loudness"],
    ["Frequency", "Cycles per second of oscillation", "Hz", "440 Hz (musical A)", "Basis of pitch and timbre"],
    ["Channel", "An independent audio stream", "-", "Left, Right", "Determines mono vs stereo processing"],
    ["Mono", "Audio with a single channel", "-", "Shape (samples,)", "Simplifies most ML pipelines"],
    ["Stereo", "Audio with two channels (L/R)", "-", "Shape (2, samples)", "Used for spatial/immersive audio"],
    ["Waveform", "Amplitude plotted against time", "-", "The Section 8 plot", "Primary visual/raw representation"],
    ["Noise", "Unwanted signal mixed with the desired signal", "-", "Hiss, hum, static", "Major challenge for robust ML models"],
    ["Silence", "Low/no signal energy over a region", "-", "Pause between words", "Used for segmentation, VAD"],
    ["Speech Signal", "Audio specifically containing spoken language", "-", "A recorded sentence", "Core subject of speech/audio research"],
    ["Duration", "Total length of the audio", "seconds", "3.250 s", "Basic property for dataset statistics"],
], columns=["Concept", "Definition", "Unit", "Example", "Research Importance"])

concept_table


## Section 27 — Why Computer Audio Basics Matter for AI-Generated Speech Detection

Both **human speech** and **AI-generated speech** are, ultimately, just digital audio signals — arrays of numbers with a sampling rate and bit depth. A detection model can potentially learn to distinguish them using differences in:

- Waveform patterns
- Spectral patterns
- Temporal patterns
- Pitch-related characteristics
- Harmonic structure
- High-frequency information
- Encoding/compression artifacts
- Noise robustness

> No single feature reliably detects AI-generated speech on its own — real systems combine many of these signals, usually learned automatically by a model rather than hand-picked.

The typical research pipeline builds upward from everything in this notebook:

```
Waveform
  |
FFT
  |
STFT
  |
Spectrogram
  |
Mel Spectrogram
  |
MFCC
  |
Self-Supervised Speech Representations
  |
Classifier
  |
AI-generated speech detection
```


## Section 28 — Automatic Audio Report

A single auto-generated report, computed entirely from your uploaded file.


In [ ]:
f0_report_line = "Pitch: Not reliably estimated (see Section 18)."
try:
    if "valid_f0" in dir() and len(valid_f0) > 0:
        f0_report_line = f"Pitch (F0): {valid_f0.min():.1f}-{valid_f0.max():.1f} Hz (mean {valid_f0.mean():.1f} Hz)"
except NameError:
    pass

report_lines = [
    "================ AUDIO REPORT ================",
    f"File               : {audio_file}",
    f"Duration           : {duration_sec:.3f} s",
    f"Sampling Rate      : {sr} Hz",
    f"Channels           : {n_channels} ({'Stereo' if is_stereo else 'Mono'})",
    f"Number of Samples  : {n_samples}",
    f"Bit Depth          : {bit_depth_str}",
    f"Min Amplitude      : {min_amp:.5f}",
    f"Max Amplitude      : {max_amp:.5f}",
    f"RMS                : {rms_amp:.5f}",
    f"Nyquist Frequency  : {nyquist:.0f} Hz",
    "",
    f"Estimated Non-Silence : {non_silent_pct:.1f}%",
    f"Estimated Silence      : {silent_pct:.1f}%",
    "",
    f0_report_line,
    "================================================",
]
report = "\n".join(report_lines)
print(report)


## Section 29 — Interactive Questions

Using the numbers computed above, try answering these yourself (all answers exist somewhere in this notebook's output):

1. What is the sampling rate of your uploaded audio? **Your answer:** _____
2. How many samples are in the audio? **Your answer:** _____
3. What is the duration? **Your answer:** _____
4. Is the audio mono or stereo? **Your answer:** _____
5. What is the Nyquist frequency? **Your answer:** _____
6. What is the maximum amplitude? **Your answer:** _____
7. What percentage is estimated to be non-silent? **Your answer:** _____
8. What does the waveform represent? **Your answer:** _____
9. What is the difference between sampling rate and bit depth? **Your answer:** _____
10. Why might MP3 compression be problematic for audio research? **Your answer:** _____


## Section 30 — Mini Practical Experiments

Try re-running this notebook (from Section 2 onward) with different uploads to explore:

1. **Human speech recording** — analyze it fully with this notebook.
2. **Speech with background noise** — compare its waveform and RMS to a clean recording.
3. **Stereo audio file** — compare Left vs Right channels (Sections 7 & 22).
4. **Mono conversion** — observe the averaged waveform (Section 23).
5. **WAV vs MP3 of the same clip** (if you have both) — compare duration, sampling rate, waveform shape, file size, and FFT spectrum.
6. **Resampling** — try the cell below to compare your original audio against 8 kHz and 16 kHz versions.

Experiment 6 is implemented below — it does **not** overwrite your original audio.


In [ ]:
resample_targets = [8000, 16000]
resampled_versions = {}

for target_sr in resample_targets:
    resampled_versions[target_sr] = librosa.resample(
        wave_for_plot.astype(np.float32), orig_sr=sr, target_sr=target_sr
    )

fig, axes = plt.subplots(1 + len(resample_targets), 1, figsize=(12, 3 * (1 + len(resample_targets))), sharex=True)

librosa.display.waveshow(wave_for_plot, sr=sr, ax=axes[0])
axes[0].set_title(f"Original ({sr} Hz)")

for ax, (target_sr, resampled) in zip(axes[1:], resampled_versions.items()):
    librosa.display.waveshow(resampled, sr=target_sr, ax=ax)
    ax.set_title(f"Resampled to {target_sr} Hz")

axes[-1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

print("Lower sampling rates lose the ability to represent higher frequencies (lower Nyquist limit),")
print("which is audible as a duller / less crisp sound, especially for sibilant speech sounds ('s', 'sh').")
print("(Original audio variable 'audio' was NOT modified.)")


## Section 31 — Complete Concept Map

```
REAL WORLD SOUND
      |            (a physical vibration in the air)
SOUND WAVE
      |            (a microphone converts pressure changes to voltage)
MICROPHONE
      |
ANALOG SIGNAL
      |            (measuring the voltage at fixed time intervals)
SAMPLING
      |            (how many measurements per second)
SAMPLING RATE
      |            (rounding each measurement to a finite precision)
QUANTIZATION
      |            (how many bits are used per measurement)
BIT DEPTH
      |            (the resulting sequence of numbers)
DIGITAL AUDIO
      |            (plotting amplitude vs time)
WAVEFORM
      |            (examining which frequencies are present)
FREQUENCY ANALYSIS
      |            (frequency content over time - next stage)
SPECTROGRAM
      |            (compact numeric summaries for models)
AUDIO FEATURES
      |            (models that learn from those features)
DEEP LEARNING
      |
AUDIO RESEARCH
```

Every arrow above is a transformation you've now seen hands-on in this notebook, using your own uploaded audio.


## Final Section — What You Should Know After This Lab

- [ ] I understand digital audio
- [ ] I understand sampling
- [ ] I understand sampling rate
- [ ] I understand Nyquist frequency
- [ ] I understand amplitude
- [ ] I understand frequency
- [ ] I understand bit depth
- [ ] I understand channels
- [ ] I understand mono/stereo
- [ ] I understand WAV/MP3
- [ ] I can read a waveform
- [ ] I can interpret RMS energy
- [ ] I understand silence
- [ ] I understand noise
- [ ] I understand speech signals
- [ ] I understand why audio preprocessing matters
- [ ] I understand the connection to AI-generated speech detection

**Next stage preview:** FFT -> STFT -> Spectrogram -> Mel-Spectrogram — the frequency-over-time representations that power most modern speech/audio deep learning models.
